## Spark Streaming and Using PostgreSQL as a Sink

### Introduction to Spark Streaming Sinks
In Spark Streaming, a sink is where the output of the stream processing is sent. The choice of sink depends on the system requirements and the use case. Common sinks include databases, files, and real-time dashboards. 

Here, we focus on using PostgreSQL as a sink.

### Setting Up Spark with JDBC for PostgreSQL
To use PostgreSQL as a sink in Spark Streaming, you need to configure the Spark session with the PostgreSQL JDBC driver. This enables Spark to send data directly to your PostgreSQL database.


### Defining Connection Properties
Define the necessary connection properties to connect to your PostgreSQL database:

```python
# Connection URL and properties for PostgreSQL
url = "jdbc:postgresql://soe-postgres.postgres.database.azure.com:5432/music_store"
postgres_options = {
    "url": url,
    "user": "student",
    "password": "reRZ2pjg1WxqlwjU",
    "driver": "org.postgresql.Driver"
}
```

Like before we'll provide the path to postgreSQL library.

In [ ]:
import os

# Path to the PostgreSQL JDBC jar file
rel_path = os.path.relpath('./libs/postgresql-42.7.3.jar')

# Connection URL and properties for PostgreSQL
url = "jdbc:postgresql://fhtw-big-data.postgres.database.azure.com/music_store"
postgres_options = {
    "url": url,
    "user": "student",
    "password": "reRZ2pjg1WxqlwjU",
    "driver": "org.postgresql.Driver"
}

In [ ]:
# Kafka configuration parameters for Confluent Cloud
kafka_params = {
      "kafka.bootstrap.servers": "46.225.20.89:9092",
      "subscribe": "fhtw-stocks",
      "kafka.security.protocol": "PLAINTEXT",
      "startingOffsets": "latest",
}

### Configure your spark session to include custom jar files

In Apache Spark, external libraries, such as JDBC drivers or connectors for other data sources like Apache Kafka, can be included in your Spark jobs by configuring the Spark session

In [ ]:
import os
from pyspark.sql import SparkSession

# Create a Spark session and configure it to use the JDBC jar
spark = SparkSession.builder \
    .appName("PostgreSQL Integration Example") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.0") \
    .config("spark.jars", rel_path) \
    .config("spark.executor.extraClassPath", rel_path) \
    .config("spark.driver.extraClassPath", rel_path) \
    .getOrCreate()

### Data Structure

The messages in the Kafka topic have the following JSON structure:

```json
{
  "side": "SELL",
  "quantity": 1587,
  "symbol": "ZVV",
  "price": 326,
  "account": "LMN456",
  "userid": "User_5"
}


## Streaming Data Processing and Writing to PostgreSQL
Once the Spark session is set up and configured, you can process streaming data and write the output directly to PostgreSQL.

### Reading from Kafka
Assuming data is being streamed from Kafka, set up the stream reading:

In [ ]:
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType

# Define the schema for the JSON data
schema = StructType([
    StructField("side", StringType(), nullable=False),
    StructField("quantity", IntegerType(), nullable=False),
    StructField("symbol", StringType(), nullable=False),
    StructField("price", FloatType(), nullable=False),
    StructField("account", StringType(), nullable=False),
    StructField("userid", StringType(), nullable=False)
])

In [ ]:
# Read streaming data from Kafka
df = spark \
    .readStream \
    .format("kafka") \
    .options(**kafka_params) \
    .load()

# Select and cast the key and value columns to STRING
df_message = df.selectExpr("CAST(key AS STRING)", "CAST(value AS STRING)")

# Parse the JSON data and apply schema
df_value = df_message.selectExpr("CAST(value AS STRING) as json_string")

parsed_df = df_value.withColumn("jsonData", 
                                from_json(col("json_string"), schema)).select("jsonData.*")

In [ ]:
# Display the total quantity
query = parsed_df.writeStream \
    .outputMode("append") \
    .format("console") \
    .start()

import time
time.sleep(30)
query.stop()

### Understanding `foreachBatch` in Spark Streaming

#### Introduction to `foreachBatch`
In Spark Structured Streaming, `foreachBatch()` is a method that allows you to perform custom processing on the output of each micro-batch of a streaming DataFrame/Dataset. It provides the flexibility to apply batch DataFrame operations that are not available in the standard streaming DataFrame operations.

#### How `foreachBatch` Works
`foreachBatch()` takes a function as an argument. This function should have two parameters: a DataFrame representing the batch data and a long type batch ID, which is unique for each micro-batch. Inside this function, you can perform any operation that is available for batch DataFrames, such as writing to databases, performing complex transformations, or even generating plots for visualization.

##### Example Usage
Here’s how you can use `foreachBatch()` to write streaming data into a PostgreSQL database:

```python
from pyspark.sql import DataFrame

def write_to_postgres(df: DataFrame, epoch_id: int):
    df.write.format("jdbc") \
        .option("url", "jdbc:postgresql://database_url:5432/database_name") \
        .option("dbtable", "target_table") \
        .option("user", "username") \
        .option("password", "password") \
        .option("driver", "org.postgresql.Driver") \
        .mode("append") \
        .save()

# Assuming 'streaming_df' is your streaming DataFrame
query = streaming_df.writeStream \
    .foreachBatch(write_to_postgres) \
    .start()

query.awaitTermination()

In [ ]:
from pyspark.sql.functions import sum

def write_to_console(df, epoch_id):
    display(df.toPandas())

# Write results to PostgreSQL using foreachBatch
query = parsed_df.writeStream \
    .outputMode("append") \
    .foreachBatch(write_to_console) \
    .start()

time.sleep(15)
query.stop()

For storing the data in postgreSQL now, all we need to do, is to stream into the database and configure the jdbc sink 

In [ ]:
# Write results to PostgreSQL using foreachBatch

student_id = "YOUR STUDENT ID" # Replace with your actual student ID!


table_name = f"streaming_data.{student_id}_stock_data"

def write_to_postgres(df, epoch_id):
    df.write \
        .format("jdbc") \
        .options(**postgres_options) \
        .option("dbtable", table_name) \
        .mode("append") \
        .save()

#### After running the next cell, look inside the database into your table using streaming_data, you should see entries being added!

In [ ]:
from pyspark.sql.functions import sum

query = parsed_df.writeStream \
    .outputMode("append") \
    .foreachBatch(write_to_postgres) \
    .start()

In [ ]:
# ALWAYS stop a stream otherwise you might flood the database
query.stop()

### Calculate the total quantity of stocks traded and store in database

#### After running the next cell, look inside the database into your table using streaming_data, you should see entries being added!

In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", "4")

In [ ]:
# Write results to PostgreSQL using foreachBatch
student_id = "YOUR STUDENT ID" # Replace with your actual student ID!
table_name = f"streaming_data.{student_id}_stock_data_quantity"

# Write results to PostgreSQL using foreachBatch
def write_to_postgres(df, epoch_id):
    df.write \
        .format("jdbc") \
        .options(**postgres_options) \
        .option("dbtable", table_name) \
        .mode("append") \
        .save()

# Calculate the total quantity of stocks traded
symbol_count_df = (parsed_df
                   .groupBy("symbol")
                   .count()
                   .withColumnRenamed("count", "trade_count"))

print(f'Starting to write to table {table_name}')

# Display the count of trades for each symbol
query = symbol_count_df.writeStream \
    .outputMode("complete") \
    .foreachBatch(write_to_postgres) \
    .start()    

In [ ]:
query.stop()

#### foreachBatch parameters
Typically, foreachBatch expects a function with exactly two arguments: the DataFrame and the epoch ID. To pass additional parameters, you can use a lambda or define a higher-order function that returns the desired function with additional parameters.

In [ ]:
def write_to_postgres(table_name):
    def inner_write_to_postgres(df, epoch_id):
        df.write \
            .format("jdbc") \
            .options(**postgres_options) \
            .option("dbtable", table_name) \
            .mode("append") \
            .save()
    return inner_write_to_postgres

In [ ]:
from pyspark.sql.functions import current_timestamp

student_id = "YOUR STUDENT ID" # Replace with your actual student ID!
table_name = f"streaming_data.{student_id}_stock_data_quantity_timestamp"

parsed_df_with_time = parsed_df.withColumn("timestamp", current_timestamp())

# Group by symbol and timestamp, then count
symbol_count_df = (parsed_df_with_time
                   .groupBy("symbol", "timestamp")
                   .count()
                   .withColumnRenamed("count", "trade_count"))

print(f'Starting to write to table {table_name}')

# Continue with your streaming write
query = symbol_count_df.writeStream \
    .outputMode("complete") \
    .foreachBatch(write_to_postgres(table_name)) \
    .start()

In [ ]:
query.stop()

In [ ]:
# stop all running spark streams
for s in spark.streams.active:
    s.stop()

In [ ]:
spark.stop()